# 03 · Embed — 04 Provider dispatch, one entry point

**Ported from `terrier-ta`'s `services/embedding.py` (166 L), branch `main`. Notebooks 01-03 each hardcode ONE embedding method. This notebook is the general version: one `embed_texts()` that dispatches to hash, OpenAI, or a local sentence-transformers model by a single setting, with an explicit choice between failing loudly and degrading quietly.**

Same shape as `02-chunk/03-store-backends.ipynb`'s upsert-by-flag, applied to
the embed side — a provider chosen at runtime rather than a notebook per
provider.

## What this notebook demonstrates

| Name | What it does | Example |
|---|---|---|
| `embedding_provider` / `embedding_model_name` / `embedding_dimension` | The provider, model and dimension resolved in ONE place | `embedding_provider()` |
| `embedding_strict` | Explicit choice: raise on failure, or degrade to the hash fallback | `embedding_strict()` |
| `embed_texts` / `embed_query` | The one entry point every caller uses, regardless of provider | `embed_texts(["some text"])` |
| `EmbeddingError` | Raised in strict mode; carries which provider/model/dim failed | `EmbeddingError(...)` |


In [ ]:
import sys
from pathlib import Path

_root = Path.cwd().resolve()
for _ in range(6):
    if (_root / "nbio.py").is_file():
        sys.path.insert(0, str(_root))
        break
    _root = _root.parent
else:
    raise RuntimeError("could not locate nbio.py above the current directory")

import nbio

repo_root = nbio.bootstrap()
nbio.show_environment(extra_keys=["EMBEDDING_PROVIDER", "EMBEDDING_STRICT"])

## Step 1 — the settings this dispatch reads, resolved from plain env vars

The donor reads these from a product `Settings` object
(`terrier_ta.config.settings`); this notebook reads the same three
questions directly from `os.environ`, since the cookbook has no settings
object to plug into. `EMBEDDING_STRICT` defaults to `True` only when
`EMBEDDING_PROVIDER=openai` and a key is actually present — the same
"strict only when it would otherwise silently degrade for no reason"
default the donor uses.

In [ ]:
import os

EMBEDDING_MODEL_DIMENSIONS = {
    "text-embedding-3-small": 1536,
    "text-embedding-3-large": 3072,
    "baai/bge-small-en-v1.5": 384,
    "baai/bge-large-en-v1.5": 1024,
}


def embedding_provider() -> str:
    return os.environ.get("EMBEDDING_PROVIDER", "hash").strip().lower()


def _has_openai_key() -> bool:
    return bool(os.environ.get("OPENAI_API_KEY", "").strip())


def embedding_model_name() -> str:
    prov = embedding_provider()
    if prov == "local":
        return os.environ.get("LOCAL_EMBEDDING_MODEL", "BAAI/bge-small-en-v1.5").strip()
    if prov == "hash":
        return "hash-v1"
    return os.environ.get("EMBEDDING_MODEL", "text-embedding-3-small").strip()


def embedding_strict() -> bool:
    if "EMBEDDING_STRICT" in os.environ:
        return os.environ["EMBEDDING_STRICT"].strip().lower() in ("1", "true", "yes")
    return _has_openai_key() and embedding_provider() == "openai"


def embedding_dimension() -> int:
    prov = embedding_provider()
    if prov == "hash":
        return int(os.environ.get("EMBEDDING_DIMENSION", "384"))
    if prov == "local":
        model = embedding_model_name().lower()
        return EMBEDDING_MODEL_DIMENSIONS.get(model, 384)
    model = embedding_model_name().lower()
    if _has_openai_key() or prov == "openai":
        return EMBEDDING_MODEL_DIMENSIONS.get(model, int(os.environ.get("EMBEDDING_DIMENSION", "1536")))
    return int(os.environ.get("EMBEDDING_DIMENSION", "384"))


print("provider: ", embedding_provider())
print("model:    ", embedding_model_name())
print("strict:   ", embedding_strict())
print("dimension:", embedding_dimension())

## Step 2 — `EmbeddingError`: what strict mode raises

Ported unchanged. Carries which provider, model and dimension were in play
when it failed, so a caller two layers up doesn't have to re-derive the
context from a bare exception message.

In [ ]:
class EmbeddingError(RuntimeError):
    """Raised when the embedding provider fails in strict mode."""

    def __init__(self, message: str, *, provider: str = "", model: str = "", dim: int = 0):
        super().__init__(message)
        self.provider = provider
        self.model = model
        self.dim = dim


err = EmbeddingError("example failure", provider="openai", model="text-embedding-3-small", dim=1536)
print(f"{err}  (provider={err.provider}, model={err.model}, dim={err.dim})")

## Step 3 — `_hash_embed`: the deterministic fallback, at the ACTIVE dimension

Ported unchanged (the donor's own comment cited an internal requirement id
here -- dropped, since a comment here should describe the code, not
another repo's tracker). The dimension comes from `embedding_dimension()`
above, not a hardcoded constant -- this is what lets the same function
serve as the strict-mode fallback for whichever provider was actually
configured.

In [ ]:
import hashlib
import math


def _hash_embed(text: str, dim: int | None = None) -> list[float]:
    d = dim or embedding_dimension()
    vec = [0.0] * d
    tokens = (text or "").lower().split()
    if not tokens:
        return vec
    for tok in tokens:
        h = int(hashlib.sha256(tok.encode("utf-8")).hexdigest(), 16)
        idx = h % d
        sign = 1.0 if (h >> 8) & 1 else -1.0
        vec[idx] += sign
    norm = math.sqrt(sum(v * v for v in vec)) or 1.0
    return [v / norm for v in vec]


vec = _hash_embed("mitochondria produce ATP")
print(f"dim={len(vec)}  norm={math.sqrt(sum(v * v for v in vec)):.4f}")

## Step 4 — `_openai_embed`: the real provider, batched

Ported unchanged (batches of 64, matching the donor). Raises
`EmbeddingError` immediately if no key is present -- `embed_texts` below
is what decides whether that propagates (strict) or triggers the hash
fallback (not strict).

In [ ]:
def _openai_embed(texts: list[str]) -> list[list[float]]:
    from openai import OpenAI

    api_key = os.environ.get("OPENAI_API_KEY", "").strip()
    if not api_key:
        raise EmbeddingError("No embedding API key configured", provider="openai")
    client = OpenAI(api_key=api_key)
    model = embedding_model_name()
    out: list[list[float]] = []
    for i in range(0, len(texts), 64):
        batch = texts[i : i + 64]
        resp = client.embeddings.create(model=model, input=batch)
        out.extend([item.embedding for item in resp.data])
    return out


try:
    _openai_embed(["test"])
except EmbeddingError as exc:
    print(f"raised cleanly, as expected with no key: {exc}")

## Step 5 — `_local_embed`: sentence-transformers, cached per kernel

Ported unchanged. `_LOCAL_MODEL_CACHE` means a local model is loaded from
disk once per kernel session, not once per call -- the difference between
one multi-second load and one per embedding batch. Raises a clear
`EmbeddingError` if `sentence-transformers` isn't installed, rather than
an import traceback from three calls deep.

In [ ]:
_LOCAL_MODEL_CACHE: dict[str, object] = {}


def _local_embed(texts: list[str]) -> list[list[float]]:
    model_name = embedding_model_name()
    if model_name not in _LOCAL_MODEL_CACHE:
        try:
            from sentence_transformers import SentenceTransformer
        except ImportError as exc:
            raise EmbeddingError(
                "EMBEDDING_PROVIDER=local requires sentence-transformers "
                "(pip install sentence-transformers)",
                provider="local",
                model=model_name,
            ) from exc
        _LOCAL_MODEL_CACHE[model_name] = SentenceTransformer(model_name)
    model = _LOCAL_MODEL_CACHE[model_name]
    vectors = model.encode(texts, normalize_embeddings=True)
    return [list(map(float, v)) for v in vectors]


try:
    _local_embed(["test"])
except EmbeddingError as exc:
    print(f"raised cleanly if sentence-transformers isn't installed: {exc}")

## Step 6 — `_warn_degraded_once`: warn once, not once per call

Ported unchanged. A pipeline that silently falls back to the hash stub on
every one of a thousand calls either says nothing (a contributor never
learns retrieval quality dropped) or logs a thousand times (noise nobody
reads). This warns exactly once per kernel session -- proven below by
calling it three times and reading `_WARNED_DEGRADED`, not by inspecting
log output by eye.

In [ ]:
import logging

log = logging.getLogger(__name__)
_WARNED_DEGRADED = False


def _warn_degraded_once(provider: str) -> None:
    global _WARNED_DEGRADED
    if not _WARNED_DEGRADED:
        log.warning(
            "embedding_degraded provider=%s -- retrieval quality reduced; "
            "set OPENAI_API_KEY or EMBEDDING_STRICT=false",
            provider,
        )
        print(f"[warned once: degraded to {provider}]")
        _WARNED_DEGRADED = True


assert _WARNED_DEGRADED is False
_warn_degraded_once("hash")
_warn_degraded_once("hash")
_warn_degraded_once("hash")
assert _WARNED_DEGRADED is True
print("confirmed: three calls, one warning printed above")

## Step 7 — `embed_texts` / `embed_query`: the one entry point

Ported unchanged. Dispatches on `embedding_provider()`; when the provider
is `openai` with no key, `strict` decides whether that's a raised
`EmbeddingError` or a quiet fallback to `_hash_embed`. Demonstrated here
in the not-strict, no-key case -- the default this cookbook ships with,
so it degrades rather than crashes on a fresh clone.

In [ ]:
def embed_texts(texts: list[str]) -> list[list[float]]:
    if not texts:
        return []
    prov = embedding_provider()
    strict = embedding_strict()
    model = embedding_model_name()
    dim = embedding_dimension()

    try:
        if prov == "openai":
            if not _has_openai_key():
                if strict:
                    raise EmbeddingError("No embedding API key configured", provider="openai", model=model, dim=dim)
                _warn_degraded_once("hash")
                return [_hash_embed(t, dim) for t in texts]
            return _openai_embed(texts)
        if prov == "local":
            return _local_embed(texts)
        if prov == "hash":
            return [_hash_embed(t, dim) for t in texts]
        raise EmbeddingError(f"Unknown EMBEDDING_PROVIDER={prov!r}", provider=prov, model=model, dim=dim)
    except EmbeddingError:
        raise
    except Exception as exc:
        if strict:
            raise EmbeddingError(f"Embedding failed ({prov}/{model}): {exc}", provider=prov, model=model, dim=dim) from exc
        _warn_degraded_once("hash")
        return [_hash_embed(t, dim) for t in texts]


def embed_query(query: str) -> list[float]:
    return embed_texts([query or ""])[0]


vectors = embed_texts(["mitochondria produce ATP", "photosynthesis converts light to chemical energy"])
print(f"{len(vectors)} vectors, dimension {len(vectors[0])}  (provider={embedding_provider()!r})")

## Where this fits

Notebooks 01-03 each hardcode one path (hash, OpenAI, a dimension-mismatch
demonstration). This notebook is what those three would look like unified
behind one setting -- `EMBEDDING_PROVIDER=hash|openai|local` -- and it's
the piece `01-modules/05-gate/` and `06-bench/` would call if a stage
ever needed "just embed this" without caring which provider answers.

## What did not come across

`terrier_ta.config.settings.get_settings()` and `EMBEDDING_MODEL_DIMENSIONS`
being product-config-driven -- replaced with plain env vars and an inline
dict, since the cookbook has no settings object. The behavior (provider,
strict-mode, dimension resolution) is unchanged; only where the values
come from differs.